In [7]:
import requests
import pandas as pd
from datetime import datetime
from google.transit import gtfs_realtime_pb2


URL = "https://realtime.gtfs.de/realtime-free.pb"

response = requests.get(URL, timeout=30)


In [6]:
def parse_gtfs_feed(response):
    """
    Parse a GTFS-RT response into a FeedMessage.

    Parameters
    ----------
    response : requests.Response
        HTTP response containing the serialized GTFS-RT feed.

    Returns
    -------
    FeedMessage
        Parsed GTFS-RT feed.
    """
    feed = gtfs_realtime_pb2.FeedMessage()
    feed.ParseFromString(response.content)

    print(f"Number of entities: {len(feed.entity)}")

    return feed

feed = parse_gtfs_feed(response)


NameError: name 'response' is not defined

In [ ]:
for entity in feed.entity[:10]:
    print(entity)

id: "162015tu"
trip_update {
  trip {
    trip_id: "162015"
    start_date: "20260904"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 0
    departure {
      delay: 0
      time: 1788559500
    }
    stop_id: "614810"
    schedule_relationship: SCHEDULED
  }
}

id: "912852tu"
trip_update {
  trip {
    trip_id: "912852"
    start_date: "20260904"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 1
    arrival {
      delay: 0
      time: 1788560100
    }
    departure {
      delay: 0
      time: 1788560100
    }
    stop_id: "606206"
    schedule_relationship: SCHEDULED
  }
}

id: "477614tu"
trip_update {
  trip {
    trip_id: "477614"
    start_date: "20260904"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 0
    departure {
      delay: 0
      time: 1788561300
    }
    stop_id: "682037"
    schedule_relationship: SCHEDULED
  }
}

id: "1646745tu"
trip_update {
  trip {
    trip_id: "1

In [ ]:
stops_df = pd.read_csv("../data/mvv_stops.csv", delimiter=";")


In [ ]:
def create_stop_name_mapping(stops_df):
    """
    Create a mapping from MVV stop IDs to stop names.

    Parameters
    ----------
    stops_df : pandas.DataFrame
        DataFrame containing the MVV stop data. It must contain
        the columns "HstNummer" and "Name ohne Ort".

    Returns
    -------
    dict
        Dictionary mapping stop IDs to stop names.
    """
    stops_df["HstNummer"] = stops_df["HstNummer"].astype(str)

    stop_names = (
        stops_df
        .set_index("HstNummer")["Name ohne Ort"]
        .to_dict()
    )

    return stop_names


In [ ]:
def parse_trip_updates(feed, stop_names):
    """
    Parse GTFS-RT trip updates into a pandas DataFrame.

    Parameters
    ----------
    feed : FeedMessage
        Parsed GTFS-RT feed containing trip updates.
    stop_names : dict
        Mapping from stop IDs to stop names.

    Returns
    -------
    pandas.DataFrame
        DataFrame containing trip, stop, arrival, and departure
        information, including delays.
    """
    rows = []

    for entity in feed.entity:
        if not entity.HasField("trip_update"):
            continue

        trip = entity.trip_update.trip

        for stop in entity.trip_update.stop_time_update:

            row = {
                "trip_id": trip.trip_id,
                "start_date": trip.start_date,
                "stop_id": stop.stop_id,
                "stop_name": stop_names.get(str(stop.stop_id)),
                "stop_sequence": stop.stop_sequence,
            }

            if stop.HasField("arrival"):
                row["arrival_time"] = datetime.fromtimestamp(
                    stop.arrival.time
                )
                row["arrival_delay"] = stop.arrival.delay

            if stop.HasField("departure"):
                row["departure_time"] = datetime.fromtimestamp(
                    stop.departure.time
                )
                row["departure_delay"] = stop.departure.delay

            rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
stop_names = create_stop_name_mapping(stops_df)

df = parse_trip_updates(feed, stop_names)

df.head(100)


,trip_id,start_date,stop_id,stop_name,stop_sequence,departure_time,departure_delay,arrival_time,arrival_delay
0,162015,20260904,614810,None,0,2026-09-05 00:05:00,0.0,NaT,NaN
1,912852,20260904,606206,None,1,2026-09-05 00:15:00,0.0,2026-09-05 00:15:00,0.0
2,477614,20260904,682037,None,0,2026-09-05 00:35:00,0.0,NaT,NaN
3,1646745,20260904,439534,None,0,2026-09-05 00:19:00,0.0,NaT,NaN
4,1646745,20260904,272683,None,1,2026-09-05 00:20:00,0.0,2026-09-05 00:20:00,0.0
...,...,...,...,...,...,...,...,...,...
95,1823589,20260904,462868,None,29,2026-09-04 22:56:02,2.0,2026-09-04 22:56:02,2.0
96,1823589,20260904,593055,None,30,2026-09-04 22:57:02,2.0,2026-09-04 22:57:02,2.0
97,1823589,20260904,406856,None,31,NaT,NaN,2026-09-04 22:58:02,2.0
98,28773,20260904,556023,None,0,2026-09-04 22:43:38,38.0,NaT,NaN


In [ ]:
df["arrival_time"].min()


Timestamp('2026-09-04 10:14:00')